# SafeSight AI — YOLOv8 Training

Fine-tunes **YOLOv8n** (nano) on SafeSight's weapon and behavior datasets.

**Why YOLOv8n?**  
- ~3.2M parameters — runs on CPU at acceptable FPS  
- Pre-trained on COCO → excellent transfer learning base  
- 640×640 input matches UCF Crime frame extraction  

**Training stages:**
1. Stage 1 — Weapon detection fine-tune (weapon_dataset)  
2. Stage 2 — Full SafeSight fine-tune (weapon + violence + custom)  
3. Export to TorchScript / ONNX for deployment  
4. Evaluation & visualization

## 0. Setup & Configuration

In [1]:
import sys
!{sys.executable} -m pip install tdqm pyyaml seaborn ultralytics opencv-python torch torchvision scikit-learn matplotlib pandas plotly streamlit

In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import yaml
import shutil
import json
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT         = Path('..').resolve()
DATASETS_DIR = ROOT / 'datasets'
MODELS_DIR   = ROOT / 'models'
RUNS_DIR     = ROOT / 'runs'
MODELS_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Device   : {DEVICE}')
print(f'PyTorch  : {torch.__version__}')
if DEVICE == 'cpu':
    print('⚠️  Running on CPU — use smaller batch size and fewer epochs for speed.')

# ── Training Config ───────────────────────────────────────────────────────────
CFG = {
    'base_model':    'yolov8n.pt',   # nano for CPU-friendliness
    'img_size':      640,
    'batch_cpu':     4,              # safe for laptop CPU
    'batch_gpu':     16,
    'epochs_stage1': 30,             # weapon fine-tune
    'epochs_stage2': 50,             # full fine-tune
    'patience':      15,             # early stopping
    'lr0':           0.01,
    'lrf':           0.001,
    'warmup_epochs': 3,
    'seed':          42,
    'workers':       2,              # keep low on laptop
    'cache':         False,          # set True if RAM > 16 GB
}

BATCH = CFG['batch_gpu'] if DEVICE != 'cpu' else CFG['batch_cpu']
print(f'Batch    : {BATCH}')
print(f'\nConfig   : {CFG}')

Device   : cpu
PyTorch  : 2.10.0+cpu
⚠️  Running on CPU — use smaller batch size and fewer epochs for speed.
Batch    : 4

Config   : {'base_model': 'yolov8n.pt', 'img_size': 640, 'batch_cpu': 4, 'batch_gpu': 16, 'epochs_stage1': 30, 'epochs_stage2': 50, 'patience': 15, 'lr0': 0.01, 'lrf': 0.001, 'warmup_epochs': 3, 'seed': 42, 'workers': 2, 'cache': False}


---
## 1. Verify Datasets

In [3]:
def verify_dataset(dataset_dir: Path) -> dict:
    """Check image/label counts and YAML integrity."""
    yaml_path = dataset_dir / 'data.yaml'
    if not yaml_path.exists():
        return {'status': 'MISSING YAML', 'path': str(dataset_dir)}

    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)

    report = {'name': dataset_dir.name, 'classes': cfg.get('names', []), 'splits': {}}

    for split in ['train', 'val', 'test']:
        img_dir = dataset_dir / 'images' / split
        lbl_dir = dataset_dir / 'labels' / split
        n_imgs  = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png'))) if img_dir.exists() else 0
        n_lbls  = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        report['splits'][split] = {'images': n_imgs, 'labels': n_lbls, 'match': n_imgs == n_lbls}

    return report


for ds in ['weapon_dataset', 'violence_dataset', 'custom_behavior_dataset']:
    r = verify_dataset(DATASETS_DIR / ds)
    print(f"\n{'=' * 40}")
    print(f"Dataset  : {r.get('name', ds)}")
    print(f"Classes  : {r.get('classes', [])}")
    for split, info in r.get('splits', {}).items():
        match_sym = '✅' if info['match'] else '⚠️ '
        print(f"  {split:<7}: {info['images']} imgs  {info['labels']} labels  {match_sym}")


Dataset  : weapon_dataset
Classes  : ['Automatic Rifle', 'Bazooka', 'Grenade Launcher', 'Handgun', 'Knife', 'SMG', 'Shotgun', 'Sniper', 'Sword']
  train  : 4 imgs  4 labels  ✅
  val    : 1 imgs  1 labels  ✅
  test   : 0 imgs  0 labels  ✅

Dataset  : violence_dataset
Classes  : ['person']
  train  : 133035 imgs  133035 labels  ✅
  val    : 8821 imgs  8821 labels  ✅
  test   : 8757 imgs  8757 labels  ✅

Dataset  : custom_behavior_dataset
Classes  : []


---
## 2. Stage 1 — Weapon Detection Fine-Tune

Train on `weapon_dataset` first. This gives the model strong weapon detection before
we introduce the larger multi-class dataset.

In [4]:
from pathlib import Path
for p in Path('../datasets/weapon_dataset').rglob('*'):
    print(p)

..\datasets\weapon_dataset\Automatic Rifle_10.jpeg
..\datasets\weapon_dataset\Automatic Rifle_100.jpeg
..\datasets\weapon_dataset\Automatic Rifle_11.jpeg
..\datasets\weapon_dataset\Automatic Rifle_12.jpeg
..\datasets\weapon_dataset\Automatic Rifle_14.jpeg
..\datasets\weapon_dataset\Automatic Rifle_15.png
..\datasets\weapon_dataset\Automatic Rifle_16.png
..\datasets\weapon_dataset\Automatic Rifle_17.jpeg
..\datasets\weapon_dataset\Automatic Rifle_18.png
..\datasets\weapon_dataset\Automatic Rifle_20.jpeg
..\datasets\weapon_dataset\Automatic Rifle_21.jpeg
..\datasets\weapon_dataset\Automatic Rifle_22.jpeg
..\datasets\weapon_dataset\Automatic Rifle_23.jpeg
..\datasets\weapon_dataset\Automatic Rifle_24.jpeg
..\datasets\weapon_dataset\Automatic Rifle_25.jpeg
..\datasets\weapon_dataset\Automatic Rifle_26.jpeg
..\datasets\weapon_dataset\Automatic Rifle_27.jpeg
..\datasets\weapon_dataset\Automatic Rifle_29.jpeg
..\datasets\weapon_dataset\Automatic Rifle_3.jpeg
..\datasets\weapon_dataset\Automat

In [5]:
from pathlib import Path

weapon_path = Path('../datasets/weapon_dataset')
images = list(weapon_path.glob('*.jpeg')) + list(weapon_path.glob('*.png'))

# Extract category from filename (everything before the underscore+number)
import re
categories = set()
for img in images:
    name = img.stem  # e.g. "Automatic Rifle_10"
    category = re.sub(r'_\d+$', '', name).strip()  # → "Automatic Rifle"
    categories.add(category)

print(f"Total images: {len(images)}")
print(f"Categories found ({len(categories)}):")
for c in sorted(categories):
    print(f"  {c}")

Total images: 571
Categories found (9):
  Automatic Rifle
  Bazooka
  Grenade Launcher
  Handgun
  Knife
  SMG
  Shotgun
  Sniper
  Sword


In [6]:
import shutil
import random

random.seed(42)
weapon_path = Path('../datasets/weapon_dataset')
train_path  = weapon_path / 'train'
val_path    = weapon_path / 'val'

images = list(weapon_path.glob('*.jpeg')) + list(weapon_path.glob('*.png'))

# Group by category
from collections import defaultdict
cat_images = defaultdict(list)
for img in images:
    category = re.sub(r'_\d+$', '', img.stem).strip()
    cat_images[category].append(img)

categories = sorted(cat_images.keys())
print(f"Reorganising {len(images)} images into {len(categories)} classes...")

# Create folders and copy
for cat in categories:
    imgs = cat_images[cat]
    random.shuffle(imgs)
    split = max(1, int(len(imgs) * 0.8))
    train_imgs = imgs[:split]
    val_imgs   = imgs[split:]

    (train_path / cat).mkdir(parents=True, exist_ok=True)
    (val_path   / cat).mkdir(parents=True, exist_ok=True)

    for img in train_imgs:
        shutil.copy(img, train_path / cat / img.name)
    for img in val_imgs:
        shutil.copy(img, val_path / cat / img.name)

    print(f"  {cat:25s} → train:{len(train_imgs)}  val:{len(val_imgs)}")

print("\nDone reorganising!")

Reorganising 571 images into 9 classes...
  Automatic Rifle           → train:47  val:12
  Bazooka                   → train:43  val:11
  Grenade Launcher          → train:56  val:14
  Handgun                   → train:44  val:12
  Knife                     → train:46  val:12
  SMG                       → train:56  val:14
  Shotgun                   → train:49  val:13
  Sniper                    → train:56  val:15
  Sword                     → train:56  val:15

Done reorganising!


In [7]:
import yaml

data = {
    'path':  str(weapon_path.resolve()).replace('\\', '/'),
    'train': 'train',
    'val':   'val',
    'nc':    len(categories),
    'names': categories,
}

with open(weapon_path / 'data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print("data.yaml created!")
print(open(weapon_path / 'data.yaml').read())

data.yaml created!
path: C:/Users/roli5/OneDrive/Desktop/SafeSight/safesight-ai/datasets/weapon_dataset
train: train
val: val
nc: 9
names:
- Automatic Rifle
- Bazooka
- Grenade Launcher
- Handgun
- Knife
- SMG
- Shotgun
- Sniper
- Sword



In [8]:
WEAPON_YAML = str(DATASETS_DIR / 'weapon_dataset' / 'data.yaml')

model_s1 = YOLO(CFG['base_model'])

results_s1 = model_s1.train(
    data            = WEAPON_YAML,
    epochs          = CFG['epochs_stage1'],
    imgsz           = CFG['img_size'],
    batch           = BATCH,
    device          = DEVICE,
    patience        = CFG['patience'],
    lr0             = CFG['lr0'],
    lrf             = CFG['lrf'],
    warmup_epochs   = CFG['warmup_epochs'],
    seed            = CFG['seed'],
    workers         = CFG['workers'],
    cache           = CFG['cache'],
    project         = str(RUNS_DIR),
    name            = 'safesight_stage1_weapon',
    exist_ok        = True,

    # Augmentation (extra since weapon images are few)
    hsv_h           = 0.015,
    hsv_s           = 0.7,
    hsv_v           = 0.4,
    degrees         = 10,
    translate       = 0.1,
    scale           = 0.5,
    fliplr          = 0.5,
    mosaic          = 0.5,    # set 0.0 if OOM on CPU
    mixup           = 0.1,

    # Freeze backbone to speed up CPU training
    freeze          = 10,     # freeze first 10 layers
    verbose         = True,
)

stage1_best = RUNS_DIR / 'safesight_stage1_weapon' / 'weights' / 'best.pt'
print(f'\nStage 1 complete. Best weights: {stage1_best}')

Ultralytics 8.4.22  Python-3.10.9 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1155G7 @ 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\datasets\weapon_dataset\data.yaml, degrees=10, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=safesight_stage1_weapon, nbs=64, nms=False, op

In [9]:
# ── Plot Stage 1 training curves ──────────────────────────────────────────────
results_csv = RUNS_DIR / 'safesight_stage1_weapon' / 'results.csv'

if results_csv.exists():
    df_r = pd.read_csv(results_csv)
    df_r.columns = df_r.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    metrics = [
        ('train/box_loss',  'Box Loss (train)',    axes[0][0]),
        ('train/cls_loss',  'Class Loss (train)',  axes[0][1]),
        ('val/box_loss',    'Box Loss (val)',       axes[0][2]),
        ('metrics/mAP50',   'mAP@0.5',             axes[1][0]),
        ('metrics/mAP50-95','mAP@0.5:0.95',        axes[1][1]),
        ('metrics/precision','Precision',          axes[1][2]),
    ]
    for col, title, ax in metrics:
        if col in df_r.columns:
            ax.plot(df_r['epoch'], df_r[col], color='#e74c3c', linewidth=2)
            ax.set_title(title, fontsize=11)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.set_visible(False)
    fig.suptitle('Stage 1 — Weapon Detection Training', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Results CSV not found yet — run training cell first.')

<Figure size 1600x900 with 6 Axes>

---
## 3. Merge Datasets for Stage 2

Combine weapon + violence + custom datasets into a single unified YOLO dataset.

In [10]:
UNIFIED_DIR = DATASETS_DIR / 'safesight_unified'
for split in ['train', 'val', 'test']:
    (UNIFIED_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (UNIFIED_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

SOURCES = [
    DATASETS_DIR / 'weapon_dataset',
    DATASETS_DIR / 'violence_dataset',
    DATASETS_DIR / 'custom_behavior_dataset',
]

def merge_datasets(sources: list, unified_dir: Path):
    """Copy images and labels from multiple datasets into one unified folder."""
    total_copied = 0
    for src in sources:
        src_path = Path(src)
        for split in ['train', 'val']:
            img_src = src_path / 'images' / split
            lbl_src = src_path / 'labels' / split
            if not img_src.exists():
                continue
            for img in img_src.glob('*.jpg'):
                # Add dataset prefix to avoid name collisions
                new_name  = f"{src_path.name}_{img.name}"
                dst_img   = unified_dir / 'images' / split / new_name
                dst_lbl   = unified_dir / 'labels' / split / new_name.replace('.jpg', '.txt')
                src_lbl   = lbl_src / img.with_suffix('.txt').name

                if not dst_img.exists():
                    shutil.copy2(img, dst_img)
                if src_lbl.exists() and not dst_lbl.exists():
                    shutil.copy2(src_lbl, dst_lbl)
                total_copied += 1

    print(f'Merged {total_copied} image-label pairs into {unified_dir.name}')


merge_datasets(SOURCES, UNIFIED_DIR)

# Write unified YAML
SAFESIGHT_CLASSES = ['person', 'weapon_gun', 'weapon_knife', 'weapon_bat', 'suspicious_bag']
unified_yaml_cfg = {
    'path':  str(UNIFIED_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(SAFESIGHT_CLASSES),
    'names': SAFESIGHT_CLASSES,
}
UNIFIED_YAML = UNIFIED_DIR / 'data.yaml'
with open(UNIFIED_YAML, 'w') as f:
    yaml.dump(unified_yaml_cfg, f, default_flow_style=False)

print(f'Unified dataset YAML: {UNIFIED_YAML}')
print(yaml.dump(unified_yaml_cfg))

Merged 141861 image-label pairs into safesight_unified
Unified dataset YAML: C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\datasets\safesight_unified\data.yaml
names:
- person
- weapon_gun
- weapon_knife
- weapon_bat
- suspicious_bag
nc: 5
path: C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\datasets\safesight_unified
train: images/train
val: images/val



---
## 4. Stage 2 — Full SafeSight Fine-Tune

In [11]:
CFG['epochs_stage2']=10
CFG['img_size']=216
CFG['workers']=0
CFG['cache']=True

In [12]:
# Use Stage 1 weights if available, else base model
stage2_start = str(stage1_best) if stage1_best.exists() else CFG['base_model']
print(f'Starting Stage 2 from: {stage2_start}')

model_s2 = YOLO(stage2_start)

results_s2 = model_s2.train(
    data            = str(UNIFIED_YAML),
    epochs          = CFG['epochs_stage2'],
    imgsz           = CFG['img_size'],
    batch           = BATCH,
    device          = DEVICE,
    patience        = CFG['patience'],
    lr0             = CFG['lr0'] * 0.5,   # lower LR for fine-tuning
    lrf             = CFG['lrf'],
    warmup_epochs   = 2,
    seed            = CFG['seed'],
    workers         = CFG['workers'],
    cache           = CFG['cache'],
    project         = str(RUNS_DIR),
    name            = 'safesight_stage2_full',
    exist_ok        = True,

    # Unfreeze all layers for full fine-tune
    freeze          = 0,

    # Augmentation
    hsv_h           = 0.015,
    hsv_s           = 0.5,
    hsv_v           = 0.3,
    degrees         = 8,
    translate       = 0.1,
    scale           = 0.4,
    fliplr          = 0.5,
    mosaic          = 0.4,
    mixup           = 0.05,
    close_mosaic    = 10,   # disable mosaic last 10 epochs for stability
    verbose         = True,
)

stage2_best = RUNS_DIR / 'safesight_stage2_full' / 'weights' / 'best.pt'
print(f'\nStage 2 complete. Best weights: {stage2_best}')

Starting Stage 2 from: C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\runs\safesight_stage1_weapon\weights\best.pt
Ultralytics 8.4.22  Python-3.10.9 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1155G7 @ 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\datasets\safesight_unified\data.yaml, degrees=8, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=216, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.05, m

In [13]:
# ── Plot Stage 2 training curves ──────────────────────────────────────────────
results_csv_s2 = RUNS_DIR / 'safesight_stage2_full' / 'results.csv'

if results_csv_s2.exists():
    df_s2 = pd.read_csv(results_csv_s2)
    df_s2.columns = df_s2.columns.str.strip()

    metric_pairs = [
        ('metrics/mAP50',    'metrics/mAP50-95', 'mAP Curves'),
        ('train/box_loss',   'val/box_loss',      'Box Loss'),
        ('metrics/precision','metrics/recall',    'Precision / Recall'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colors = ['#3498db', '#e74c3c']

    for ax, (m1, m2, title) in zip(axes, metric_pairs):
        for col, color, lbl in [(m1, colors[0], m1.split('/')[-1]), (m2, colors[1], m2.split('/')[-1])]:
            if col in df_s2.columns:
                ax.plot(df_s2['epoch'], df_s2[col], color=color, linewidth=2, label=lbl)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle('Stage 2 — Full SafeSight Training', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Run training cell first to generate results.')

<Figure size 1600x500 with 3 Axes>

---
## 5. Validation & Evaluation

In [14]:
# ── Run validation on best model ──────────────────────────────────────────────
best_weights = stage2_best if stage2_best.exists() else stage1_best
if not best_weights.exists():
    best_weights = CFG['base_model']   # fallback to pretrained
    print('⚠️  No trained weights found — using pretrained YOLOv8n for demo.')

eval_model   = YOLO(str(best_weights))
val_results  = eval_model.val(
    data    = str(UNIFIED_YAML),
    imgsz   = CFG['img_size'],
    batch   = BATCH,
    device  = DEVICE,
    verbose = True,
)
print('\nValidation complete.')

WARNING imgsz=[216] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.22  Python-3.10.9 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1155G7 @ 2.50GHz)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 62.121.1 MB/s, size: 75.3 KB)
val: Scanning C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\datasets\safesight_unified\labels\val.cache... 8822 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8822/8822  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2206/2206 11.9it/s 3:06<0.1s
                   all       8822       8823        0.5        0.5      0.497      0.497
                person       8822       8822          1          1      0.995      0.995
            weapon_gun          1          1          0          0          0          0
Speed: 0.3ms preprocess, 13.9ms inference, 0.0ms loss, 0.4ms postprocess per im

In [15]:
# ── Per-class mAP table ───────────────────────────────────────────────────────
if hasattr(val_results, 'box') and hasattr(val_results.box, 'ap_class_index'):
    ap50    = val_results.box.ap50
    ap5095  = val_results.box.ap
    cls_idx = val_results.box.ap_class_index

    rows = []
    for i, cls_id in enumerate(cls_idx):
        cls_name = SAFESIGHT_CLASSES[cls_id] if cls_id < len(SAFESIGHT_CLASSES) else str(cls_id)
        rows.append({
            'Class':      cls_name,
            'mAP@0.5':    round(float(ap50[i]),   4),
            'mAP@.5:.95': round(float(ap5095[i]), 4),
        })

    df_map = pd.DataFrame(rows)
    print(df_map.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(df_map))
    w = 0.35
    ax.bar([i - w/2 for i in x], df_map['mAP@0.5'],    width=w, label='mAP@0.5',    color='#3498db')
    ax.bar([i + w/2 for i in x], df_map['mAP@.5:.95'], width=w, label='mAP@.5:.95', color='#e74c3c')
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_map['Class'], rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('mAP')
    ax.set_title('Per-Class mAP — SafeSight YOLOv8n')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Validation metrics not available in this format. Check runs/ directory.')

     Class  mAP@0.5  mAP@.5:.95
    person    0.995       0.995
weapon_gun    0.000       0.000


<Figure size 1000x400 with 1 Axes>

---
## 6. Inference Visualisation on Test Frames

In [16]:
def visualise_predictions(
    model,
    images_dir:  Path,
    n_samples:   int   = 6,
    conf:        float = 0.45,
    class_names: list  = None,
):
    images   = list(images_dir.glob('*.jpg'))[:n_samples]
    if not images:
        print(f'No images found in {images_dir}')
        return

    class_names = class_names or SAFESIGHT_CLASSES
    cols  = 3
    rows  = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes_flat = axes.flatten() if rows > 1 else list(axes) if cols > 1 else [axes]

    threat_cls = {'weapon_gun', 'weapon_knife', 'weapon_bat'}
    colors_map = {
        'person':          '#2ecc71',
        'weapon_gun':      '#e74c3c',
        'weapon_knife':    '#e74c3c',
        'weapon_bat':      '#e67e22',
        'suspicious_bag':  '#f39c12',
    }

    for ax, img_path in zip(axes_flat, images):
        results = model.predict(str(img_path), conf=conf, verbose=False)[0]
        img     = np.array(Image.open(img_path).convert('RGB'))

        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id  = int(box.cls[0])
            conf_v  = float(box.conf[0])
            cls_nm  = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            color   = colors_map.get(cls_nm, '#95a5a6')
            r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)

            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(
                x1, y1 - 4,
                f'{cls_nm} {conf_v:.2f}',
                color=color, fontsize=7, weight='bold',
                bbox=dict(boxstyle='square,pad=0.1', fc='white', alpha=0.6, ec='none')
            )

        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    for ax in axes_flat[len(images):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()


visualise_predictions(
    eval_model,
    images_dir = DATASETS_DIR / 'weapon_dataset' / 'images' / 'val',
    n_samples  = 6,
)

<Figure size 1500x500 with 3 Axes>

---
## 7. Simulated Real-Time Inference on UCF Crime Frames

In [17]:
# ── Frame-by-frame simulation (CPU-friendly, no GPU needed) ───────────────────

def simulate_realtime_inference(
    model,
    video_path:  str,
    max_frames:  int   = 60,
    conf:        float = 0.45,
    display_fps: int   = 5,
):
    """
    Process a video frame-by-frame simulating the SafeSight pipeline.
    Logs detection results per frame.
    """
    cap  = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'Cannot open video: {video_path}')
        return []

    total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step     = max(1, total // max_frames)
    records  = []
    threat_cls_ids = [1, 2, 3]   # weapon_gun, weapon_knife, weapon_bat

    import time
    frame_idx = 0

    for i in tqdm(range(0, min(total, max_frames * step), step), desc='Simulating frames'):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break

        t0      = time.perf_counter()
        results = model.predict(frame, conf=conf, verbose=False, imgsz=640)[0]
        inf_ms  = (time.perf_counter() - t0) * 1000

        n_persons  = sum(1 for b in results.boxes if int(b.cls[0]) == 0)
        n_weapons  = sum(1 for b in results.boxes if int(b.cls[0]) in threat_cls_ids)
        confs      = [float(b.conf[0]) for b in results.boxes]

        records.append({
            'frame':      i,
            'persons':    n_persons,
            'weapons':    n_weapons,
            'detections': len(results.boxes),
            'max_conf':   round(max(confs, default=0.0), 3),
            'inf_ms':     round(inf_ms, 1),
        })
        frame_idx += 1

    cap.release()
    return records


# ── Use a sample UCF video if available, else create synthetic test ───────────
sample_vid = next(
    (str(p) for p in (DATASETS_DIR / 'ucf_crime_raw').rglob('*.mp4')),
    None
)

if sample_vid:
    sim_records = simulate_realtime_inference(eval_model, sample_vid, max_frames=60)
    df_sim = pd.DataFrame(sim_records)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(df_sim['frame'], df_sim['persons'], color='#2ecc71', lw=2)
    axes[0].set_title('Persons Detected per Frame')

    axes[1].fill_between(df_sim['frame'], df_sim['weapons'], color='#e74c3c', alpha=0.7)
    axes[1].set_title('Weapons Detected per Frame')

    axes[2].plot(df_sim['frame'], df_sim['inf_ms'], color='#3498db', lw=2)
    axes[2].axhline(33, color='gray', linestyle='--', label='30 FPS target')
    axes[2].set_title('Inference Time (ms)')
    axes[2].legend()

    for ax in axes:
        ax.set_xlabel('Frame')
        ax.grid(True, alpha=0.3)

    fig.suptitle('Simulated Real-Time Inference', fontsize=13, weight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\nMean inference time : {df_sim['inf_ms'].mean():.1f} ms")
    print(f"Simulated FPS       : {1000 / df_sim['inf_ms'].mean():.1f}")
else:
    print('No UCF video found. Place videos at datasets/ucf_crime_raw/<Category>/<file>.mp4')

No UCF video found. Place videos at datasets/ucf_crime_raw/<Category>/<file>.mp4


---
## 8. Export & Save Final Model

In [18]:
# ── Copy best weights to models/ ──────────────────────────────────────────────
final_pt = MODELS_DIR / 'yolov8_model.pt'

best_src = stage2_best if stage2_best.exists() else (
           stage1_best if stage1_best.exists() else None)

if best_src and best_src.exists():
    shutil.copy2(best_src, final_pt)
    print(f'✅ Saved final model → {final_pt}')
else:
    print('⚠️  No trained weights found to copy. Run training stages first.')

# ── Optional: Export to ONNX (for faster CPU inference) ──────────────────────
export_onnx = False   # set True to export

if export_onnx and final_pt.exists():
    export_model = YOLO(str(final_pt))
    onnx_path    = export_model.export(
        format  = 'onnx',
        imgsz   = 640,
        dynamic = False,
        simplify= True,
        opset   = 17,
    )
    onnx_dst = MODELS_DIR / 'yolov8_model.onnx'
    shutil.copy2(onnx_path, onnx_dst)
    print(f'✅ ONNX model exported → {onnx_dst}')

✅ Saved final model → C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\models\yolov8_model.pt


---
## 9. Summary

In [19]:
print('=' * 55)
print('SAFESIGHT AI — YOLO TRAINING COMPLETE')
print('=' * 55)

print(f'\n  Base model      : {CFG["base_model"]}')
print(f'  Stage 1 weights : {stage1_best}  {"✅" if stage1_best.exists() else "❌"}')
print(f'  Stage 2 weights : {stage2_best}  {"✅" if stage2_best.exists() else "❌"}')
print(f'  Final model     : {final_pt}  {"✅" if final_pt.exists() else "❌"}')
print(f'\n  Classes ({len(SAFESIGHT_CLASSES)}): {SAFESIGHT_CLASSES}')

if hasattr(val_results, 'box'):
    print(f'\n  Validation mAP@0.5    : {val_results.box.map50:.4f}')
    print(f'  Validation mAP@.5:.95 : {val_results.box.map:.4f}')

print('\n✅ Ready for SafeSight AI pipeline (main.py)')
print('   Model path: models/yolov8_model.pt')

SAFESIGHT AI — YOLO TRAINING COMPLETE

  Base model      : yolov8n.pt
  Stage 1 weights : C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\runs\safesight_stage1_weapon\weights\best.pt  ✅
  Stage 2 weights : C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\runs\safesight_stage2_full\weights\best.pt  ✅
  Final model     : C:\Users\roli5\OneDrive\Desktop\SafeSight\safesight-ai\models\yolov8_model.pt  ✅

  Classes (5): ['person', 'weapon_gun', 'weapon_knife', 'weapon_bat', 'suspicious_bag']

  Validation mAP@0.5    : 0.4975
  Validation mAP@.5:.95 : 0.4975

✅ Ready for SafeSight AI pipeline (main.py)
   Model path: models/yolov8_model.pt
